# Experiment 4 — Stage A Extraction (LR only)

The two-stage QSBC pipeline, tested linearly:

- **Stage A (extraction):** ridge linear model, X (TF-IDF) → scores over
  the 1,408 shared ideas; take top-k → B-hat.
- **Stage B (probe):** logistic regression on X + B-hat → Y (same config
  as exp2/exp3, same 60/40 doc split, seed 42).

References from exp3: X-only floor **0.656**, golden shared-B ceiling
**0.747** (+9.1pt). The question: how much of that lift does a *linear,
label-blind* extractor recover?

Note: B-hat is generated for train AND test by the same extractor (trained
on train docs only), so Stage B sees realistic (noisy) C at test time.


In [ ]:
# 1. Setup + fetch committed inputs
import os
os.chdir('/content')
if not os.path.exists('/content/qsbc'):
    !git clone -q https://github.com/jpeckenpaugh/qsbc.git qsbc
os.chdir('/content/qsbc')
print('cwd:', os.getcwd())
!pip install -q pandas scikit-learn


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('results/exp1/samples_2858.csv')
df = df.drop_duplicates(subset=['sentence_id']).reset_index(drop=True)
ideas_shared = pd.read_csv('results/exp3/sample_ideas_shared.csv')

grp = ideas_shared.groupby('sentence_id')['idea_id'].apply(list).to_dict()
df['B'] = df['sentence_id'].map(grp).fillna('').apply(
    lambda x: x if isinstance(x, list) else [])
print('samples:', len(df), '| shared-idea space:', ideas_shared['idea_id'].nunique())


In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
docs = df['doc_key'].unique()
train_docs, test_docs = train_test_split(docs, test_size=0.4, random_state=SEED)
train = df[df['doc_key'].isin(train_docs)].reset_index(drop=True)
test  = df[df['doc_key'].isin(test_docs)].reset_index(drop=True)
print('doc overlap:', len(set(train_docs) & set(test_docs)))
print('train:', len(train), '| test:', len(test))
print('train classes:', train['label'].nunique(), '| test classes:', test['label'].nunique())


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import hstack

tf = TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2),
                     stop_words='english')
Xtr = tf.fit_transform(train['sentence'])
Xte = tf.transform(test['sentence'])
print('X dims:', Xtr.shape)

mlb = MultiLabelBinarizer()
mlb.fit(list(train['B']) + list(test['B']))
Ytr = mlb.transform(list(train['B'])).astype(np.float64)   # golden shared-B targets
Yte = mlb.transform(list(test['B'])).astype(np.float64)
print('C dims (shared ideas):', Ytr.shape[1])


In [ ]:
import numpy as np

# --- Stage A: ridge linear extractor (shared solve across all C outputs) ---
ALPHA = 1.0
K = 3   # derive the top-3 most relevant ideas per sentence

Xd = Xtr.toarray()
d = Xd.shape[1]
A = Xd.T @ Xd + ALPHA * np.eye(d)
W = np.linalg.solve(A, Xd.T @ Ytr)      # (d, nC): one linear weight per idea
S_tr = Xd @ W
S_te = Xte.toarray() @ W

def topk(S, k):
    return [list(r) for r in np.argsort(-S, axis=1)[:, :k]]

Btr = topk(S_tr, K)
Bte = topk(S_te, K)

Chtr = np.zeros_like(Ytr); Chte = np.zeros_like(Yte)
for i, row in enumerate(Btr): Chtr[i, row] = 1
for i, row in enumerate(Bte): Chte[i, row] = 1
print('derived B-hat: top-%d per sentence' % K)


In [ ]:
def retrieval(S, Y, B_, k):
    """precision@k and recall@k of derived B-hat vs golden shared-B."""
    hits = pos = 0
    for i, row in enumerate(B_):
        gold = set(np.where(Y[i] > 0)[0])
        if not gold:
            continue
        hits += len(set(row) & gold)
        pos += len(gold)
    return hits / (len(B_) * k), hits / max(pos, 1)

p_tr, r_tr = retrieval(S_tr, Ytr, Btr, K)
p_te, r_te = retrieval(S_te, Yte, Bte, K)
print(f'Stage A retrieval (k={K}):  precision@k train={p_tr:.3f} test={p_te:.3f} | recall@k train={r_tr:.3f} test={r_te:.3f}')


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# --- Stage B: logistic probe on X + derived B-hat -> Y ---
lr = LogisticRegression(max_iter=1000, C=10, class_weight='balanced')
lr.fit(hstack([Xtr, Chtr]), train['label'])
p = lr.predict(hstack([Xte, Chte]))
accB = accuracy_score(test['label'], p)
f1B = f1_score(test['label'], p, average='macro')
print('Stage B (X + derived B-hat -> Y):  acc=%.4f  macro-F1=%.4f' % (accB, f1B))


In [ ]:
import pandas as pd
rows = [
    ['A: X -> Y (floor, exp3)', 0.655856, 0.658133],
    ['Stage B: X + derived B-hat -> Y', accB, f1B],
    ['C: X + golden shared-B -> Y (ceiling, exp3)', 0.746847, 0.748841],
]
print(pd.DataFrame(rows, columns=['Condition', 'Accuracy', 'Macro-F1']).to_string(index=False))
print('\nGap to ceiling:', round(0.746847 - accB, 4), '| Lift over floor:', round(accB - 0.655856, 4))
